# Copula Project


## 1. Data Loading

The data has been processed in `data.py` module so we can directly load it from the CSV file.

In [47]:
import pandas as pd
data_file = "../data/sp500_log_returns.csv"

data = pd.read_csv(data_file, index_col=0, parse_dates=True)

data = data.fillna(0)



## 2. Generate simulated return rates

We fit copula using rolling windows of 10 years and refit every 1 year as the model takes time to fit.

In [48]:
# The code is in `main_copula.py` module in which we combined marginal fitting, copula fitting, and simulation.
from copula_model.main_copula import main_copula

sim_days = 253
n_paths = 1000
train_days = 252 * 10
refit_days = 253
all_simulated_returns, model_info, all_independent_returns = main_copula(data, n_paths=n_paths, sim_days=sim_days, 
                                                                         random_state=42, train_days=train_days, refit_freq=refit_days)

starting distribution fitting...
starting distribution fitting...starting distribution fitting...

starting distribution fitting...
starting distribution fitting...
starting distribution fitting...
Time elapsed: 19.514872074127197 seconds
starting copula fitting...
Time elapsed: 19.59998321533203 seconds
starting copula fitting...
Time elapsed: 19.692352056503296 seconds
starting copula fitting...
Time elapsed: 19.790016889572144 seconds
starting copula fitting...
Time elapsed: 19.845102071762085 seconds
starting copula fitting...
Time elapsed: 20.076768159866333 seconds
starting copula fitting...
Correlation matrix computed in 92.01558208465576 seconds
Gaussian log-likelihood computed in 0.055541038513183594 seconds
Correlation matrix computed in 92.38616800308228 seconds
Correlation matrix computed in 92.59832191467285 seconds
Gaussian log-likelihood computed in 0.056765079498291016 seconds
Gaussian log-likelihood computed in 0.05703926086425781 seconds


/Users/macaulay/Developer/copula_model/copula_model/copula_fitting.py:162: RuntimeWarning: invalid value encountered in log
  return n_params * np.log(log_likelihood) - 2 * log_likelihood
/Users/macaulay/Developer/copula_model/copula_model/copula_fitting.py:162: RuntimeWarning: invalid value encountered in log
  return n_params * np.log(log_likelihood) - 2 * log_likelihood
/Users/macaulay/Developer/copula_model/copula_model/copula_fitting.py:162: RuntimeWarning: invalid value encountered in log
  return n_params * np.log(log_likelihood) - 2 * log_likelihood


Correlation matrix computed in 92.9799268245697 seconds
Gaussian log-likelihood computed in 0.0560450553894043 seconds
Correlation matrix computed in 93.26377701759338 seconds
Gaussian log-likelihood computed in 0.05513811111450195 seconds
Correlation matrix computed in 93.02393794059753 seconds


/Users/macaulay/Developer/copula_model/copula_model/copula_fitting.py:162: RuntimeWarning: invalid value encountered in log
  return n_params * np.log(log_likelihood) - 2 * log_likelihood
/Users/macaulay/Developer/copula_model/copula_model/copula_fitting.py:162: RuntimeWarning: invalid value encountered in log
  return n_params * np.log(log_likelihood) - 2 * log_likelihood
/Users/macaulay/Developer/copula_model/copula_model/copula_fitting.py:162: RuntimeWarning: invalid value encountered in log
  return n_params * np.log(log_likelihood) - 2 * log_likelihood


Gaussian log-likelihood computed in 0.05491495132446289 seconds
t copula fitting computed in 18.79796290397644 seconds
t copula log-likelihood computed in 1.12510085105896 seconds
Selected copula: t
time elapsed: 113.00198173522949 seconds
starting simulation...
t copula fitting computed in 21.17241382598877 seconds
t copula log-likelihood computed in 1.006788969039917 seconds
Selected copula: t
time elapsed: 115.49818325042725 seconds
starting simulation...
t copula fitting computed in 27.69685196876526 seconds
t copula log-likelihood computed in 1.005599021911621 seconds
Selected copula: t
time elapsed: 121.73848795890808 seconds
starting simulation...
t copula fitting computed in 29.71744990348816 seconds
t copula log-likelihood computed in 1.032174825668335 seconds
Selected copula: t
time elapsed: 123.19262313842773 seconds
starting simulation...
t copula fitting computed in 34.54737687110901 seconds
t copula log-likelihood computed in 1.0337412357330322 seconds
Selected copula: t


In [49]:
import pickle

with open("../results/simulated_returns.pkl", "wb") as f:
    pickle.dump({
        "all_simulated_returns": all_simulated_returns,
        "model_info": model_info,
        "all_independent_returns": all_independent_returns
    }, f)

## 3. Backtesting VaR

We perform backtesting on the simulated return rates to evaluate the accuracy of the VaR estimates. Every 10 days, we calculate the $VaR_{99\%, 10 days}$ and $VaR_{95\%, 10 days}$ based on the simulated return rates and compare them with the actual returns to count the number of exceptions.

Also, we simulated return rates assuming independence among assets as a benchmark for comparison.


In [50]:
import pickle
with open("../results/simulated_returns.pkl", "rb") as f:
    results = pickle.load(f)
all_simulated_returns = results["all_simulated_returns"]
model_info = results["model_info"]
all_independent_returns = results["all_independent_returns"]


In [60]:
# Analyze the simulated returns
import numpy as np
for key in model_info.keys():
    # print(f"fitting date: {key}, copula: {model_info[key]['copula_name']}, nu: {model_info[key].get('nu', 'N/A')}")
    simulated_matrix = all_independent_returns[key]
    simulated_matrix_copula = all_simulated_returns[key]

    portfolio_returns = np.sum(simulated_matrix, axis=2) / simulated_matrix.shape[2]
    portfolio_returns_copula = np.sum(simulated_matrix_copula, axis=2) / simulated_matrix_copula.shape[2]

    low_quantile = 0.1
    high_quantile = 0.9
    independent_var_low = np.quantile(portfolio_returns, low_quantile)
    independent_var_high = np.quantile(portfolio_returns, high_quantile)
    copula_var_low = np.quantile(portfolio_returns_copula, low_quantile)
    copula_var_high = np.quantile(portfolio_returns_copula, high_quantile)

    print(f"fitting date: {key}, \n independent 1% quantile: {independent_var_low:.5f}, copula 1% quantile: {copula_var_low:.5f}, \n independent 99% quantile: {independent_var_high:.5f}, copula 99% quantile: {copula_var_high:.5f}")

fitting date: 2020-01-08 00:00:00, 
 independent 1% quantile: -0.00037, copula 1% quantile: -0.01014, 
 independent 99% quantile: 0.00180, copula 99% quantile: 0.01154
fitting date: 2021-01-08 00:00:00, 
 independent 1% quantile: -0.00065, copula 1% quantile: -0.01148, 
 independent 99% quantile: 0.00219, copula 99% quantile: 0.01308
fitting date: 2022-01-10 00:00:00, 
 independent 1% quantile: -0.00051, copula 1% quantile: -0.01067, 
 independent 99% quantile: 0.00214, copula 99% quantile: 0.01234
fitting date: 2023-01-12 00:00:00, 
 independent 1% quantile: -0.00061, copula 1% quantile: -0.01175, 
 independent 99% quantile: 0.00219, copula 99% quantile: 0.01335
fitting date: 2024-01-17 00:00:00, 
 independent 1% quantile: -0.00067, copula 1% quantile: -0.01213, 
 independent 99% quantile: 0.00214, copula 99% quantile: 0.01363
fitting date: 2025-01-21 00:00:00, 
 independent 1% quantile: -0.00063, copula 1% quantile: -0.01224, 
 independent 99% quantile: 0.00218, copula 99% quantile: 

In [ ]:
# Analyze the model info
for key in model_info.keys():
    print(f"fitting date: {key}, copula: {model_info[key]['copula_name']}, nu: {model_info[key].get('nu', 'N/A')}")


fitting date: 2020-01-08 00:00:00, copula: t, nu: 9.81936352241863
fitting date: 2021-01-08 00:00:00, copula: t, nu: 8.064782590961087
fitting date: 2022-01-10 00:00:00, copula: t, nu: 7.958202257490866
fitting date: 2023-01-12 00:00:00, copula: t, nu: 8.587323507169307
fitting date: 2024-01-17 00:00:00, copula: t, nu: 9.724872044082682
fitting date: 2025-01-21 00:00:00, copula: t, nu: 9.529783506131793


In [54]:
# Analyze the model info
for key in model_info.keys():
    corr_matrix = model_info[key]['corr_matrix']
    print(f"fitting date: {key}, correlation matrix:\n{corr_matrix}\n")

fitting date: 2020-01-08 00:00:00, correlation matrix:
[[1.         0.25135828 0.22668861 ... 0.2755526  0.36510618 0.18096865]
 [0.25135828 1.         0.19294253 ... 0.22807553 0.32770435 0.11335533]
 [0.22668861 0.19294253 1.         ... 0.21777834 0.31088217 0.45800343]
 ...
 [0.2755526  0.22807553 0.21777834 ... 1.         0.3645142  0.15124268]
 [0.36510618 0.32770435 0.31088217 ... 0.3645142  1.         0.28515786]
 [0.18096865 0.11335533 0.45800343 ... 0.15124268 0.28515786 1.        ]]

fitting date: 2021-01-08 00:00:00, correlation matrix:
[[1.         0.26263353 0.21670038 ... 0.26932472 0.33672786 0.16073783]
 [0.26263353 1.         0.23304742 ... 0.29520966 0.41469945 0.16328013]
 [0.21670038 0.23304742 1.         ... 0.2171963  0.30843546 0.47078713]
 ...
 [0.26932472 0.29520966 0.2171963  ... 1.         0.37279705 0.15226454]
 [0.33672786 0.41469945 0.30843546 ... 0.37279705 1.         0.26804008]
 [0.16073783 0.16328013 0.47078713 ... 0.15226454 0.26804008 1.        ]]

